# 40 — JD Parsing
**Goal:** Parse job description structure into sections.

A job description is a single blob of prose, but it is *internally structured*: a header (title, company), an "about" pitch, a responsibilities list, a qualifications list, a nice-to-have list, and a benefits block. This chapter turns that blob into labeled buckets — the JD-side mirror of the resume-section parsing done in earlier blocks. Everything that follows in Block G assumes these buckets exist: skill extraction, responsibility detection, and qualification mining all read from *specific* sections rather than the raw text.

**Why it matters for resumes / ATS:** an ATS compares two documents — the resume and the JD — and every comparison is only as precise as the sections it compares. A skill in *Qualifications* is a hard requirement; the same skill in *Nice to have* is a soft bonus. If the parser cannot tell the two apart, every downstream score is wrong. Section parsing is the load-bearing wall for the whole matching pipeline.

## 1. Typical JD Structure

Almost every JD follows the same skeleton: **header** (title, company, location), **about the role** (why the team exists), **responsibilities** (what the hire will do), **qualifications** (what they must already have), **nice to have** (soft extras), and **benefits** (compensation/perks). The wording varies — "What you'll do", "Requirements", "Perks" — but the *order* is remarkably stable, which is exactly what makes heading-based detection feasible.

**What the code does:** defines `jd`, a sample senior-data-scientist posting that will be reused by every later chapter in this block (Ch. 41–45 all operate on this same string). The `print` statement simply spells out the canonical section order the detector must recover. Working from one controlled example makes it easy to see where the parser succeeds and where it slips.

In [ ]:
jd = """Senior Data Scientist
Google, Mountain View

About the role
We are looking for a senior data scientist to join our ML team.

Responsibilities
- Develop and deploy ML models at scale
- Collaborate with product teams
- Mentor junior data scientists

Qualifications
- 5+ years experience in data science
- Strong Python and SQL skills
- Experience with TensorFlow or PyTorch
- MS/PhD in Computer Science or related field

Nice to have
- Experience with NLP
- Published research papers

Benefits
- Competitive salary and equity
- Health insurance
- Remote work options
"""
print("JD structure: Title -> Company -> About -> Responsibilities -> Qualifications -> Nice-to-have -> Benefits")

## 2. JD Section Detection

Section detection is a **keyword-triggered state machine**: walk the text line by line; when a line contains a heading keyword ("responsibilities", "nice to have", …) *and* is short (`len(ls) < 40`), switch the active section; otherwise append the line to the current section. The length guard is the trick that stops body sentences like "We are looking for a senior data scientist…" from being mistaken for headings.

**What the code does:**
- `JD_SECTIONS` maps canonical section names to lists of heading synonyms ("what you'll do", "what we offer", "experience required").
- `detect_jd_sections()` lowercases each line, scans the synonym lists in order, and resets `current_section` when a heading matches.
- Non-heading lines accumulate via `setdefault` into the open section; lines before the first heading land in `header`.

**Expected:** running this on the sample `jd` recovers `header` (title + company), `about`, `responsibilities`, `qualifications`, `nice_to_have`, and `benefits` buckets. One sharp edge: the line "Strong Python and SQL skills" *contains* the keyword "skills", so a mid-list bullet can retrigger the qualifications heading and reset the bucket — real-world JD parsers need to require headings to be short, capitalized, and dash-free.

In [ ]:
import re

JD_SECTIONS = {
    "about": ["about", "overview", "summary", "who we are", "the role"],
    "responsibilities": ["responsibilities", "what you'll do", "the role", "key duties", "what you will do"],
    "qualifications": ["qualifications", "requirements", "what you bring", "skills", "experience required"],
    "nice_to_have": ["nice to have", "bonus", "preferred", "plus", "good to have"],
    "benefits": ["benefits", "perks", "what we offer", "compensation"],
}

def detect_jd_sections(text):
    lines = text.split("\n")
    sections = {}
    current_section = "header"
    sections[current_section] = []
    for line in lines:
        ls = line.strip().lower()
        found = False
        for sec_name, keywords in JD_SECTIONS.items():
            if any(kw in ls for kw in keywords) and len(ls) < 40:
                current_section = sec_name
                sections[current_section] = []
                found = True
                break
        if not found and ls:
            sections.setdefault(current_section, []).append(line.strip())
    return sections

secs = detect_jd_sections(jd)
for name, content in secs.items():
    print(f"\n[{name.upper()}]")
    for line in content[:3]:
        print(f"  {line[:60]}")

## Summary: JD parsing identifies sections for targeted extraction. Similar approach to resume sections.

**Sections turn a prose JD into addressable buckets, and every later chapter depends on those buckets being correct.**

The same divide-and-conquer idea used for resumes applies here: parse structure first, extract content second. With sections labeled, Ch. 41 can pull skills out of *Qualifications* vs *Nice to have* separately, Ch. 42 knows where the duties live, and Ch. 43 knows where to hunt for degrees and years. The trade-off shown in this chapter is classic rules-vs-robustness: keyword+length heuristics are cheap and explainable, but a bullet containing a heading word can silently corrupt a bucket. This feeds directly into the skill extraction of Ch. 41.